# Notebook 03 — Condition B: XGBoost with Missingness-Aware Preprocessing

## Requires
- `data/raw/`

## Produces
- `results/models/xgb_condition_B.pkl`
- Row appended to `results/experiment_log.csv`

## Feature engineering
Lag features added after split, before imputation. Strategy B then adds missingness indicators for the 40 ORIGINAL features only (not lag features). Total: 40 original + 48 lag + ~36 indicators ≈ 124 features.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from src.config import RANDOM_SEED, DATA_DIR, MODELS_DIR, ALL_FEATURES
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers
from src.features import add_lag_features
from src.utils import set_all_seeds, validate_no_nans, create_patient_splits
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
print('Imports OK')

## 1. Load Data and Create Splits

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Train: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}')

## 2. Lag Feature Engineering

In [ ]:
# Per-patient groupby — no cross-patient leakage
train_df = add_lag_features(train_df)
val_df   = add_lag_features(val_df)
test_df  = add_lag_features(test_df)
print(f'Columns after lag features: {len(train_df.columns)}')

## 3. Strategy B Preprocessing (forward-fill + missingness indicators + scaling)

In [ ]:
meta_cols      = {'patient_id', 'hospital_id', 'SepsisLabel', 'EarlyLabel'}
original_feats = [c for c in ALL_FEATURES if c in train_df.columns]

def add_indicators_and_ffill(df):
    """Add missingness indicators for ORIGINAL features only, then forward-fill all."""
    df = df.copy()
    all_feat_cols = [c for c in df.columns if c not in meta_cols]
    for feat in original_feats:            # indicators for original 40 only
        if df[feat].isna().any():
            df[f'{feat}_missing'] = df[feat].isna().astype('int8')
    df[all_feat_cols] = df.groupby('patient_id')[all_feat_cols].ffill()
    return df

train_proc = add_indicators_and_ffill(train_df)
val_proc   = add_indicators_and_ffill(val_df)
test_proc  = add_indicators_and_ffill(test_df)

feature_cols_B = [c for c in train_proc.columns if c not in meta_cols]
for proc_df in (val_proc, test_proc):      # align columns across splits
    for col in feature_cols_B:
        if col not in proc_df.columns:
            proc_df[col] = 0

X_train_raw = train_proc[feature_cols_B].values
X_val_raw   = val_proc[feature_cols_B].values
X_test_raw  = test_proc[feature_cols_B].values
y_train = train_df['EarlyLabel'].values
y_val   = val_df['EarlyLabel'].values
y_test  = test_df['EarlyLabel'].values

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train_raw)
X_val   = imputer.transform(X_val_raw)
X_test  = imputer.transform(X_test_raw)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

for arr, name in [(X_train,'train_B'),(X_val,'val_B'),(X_test,'test_B')]:
    validate_no_nans(arr, name, feature_cols_B)

os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(imputer,        f'{MODELS_DIR}strategy_B_lag_imputer.pkl')
joblib.dump(scaler,         f'{MODELS_DIR}strategy_B_lag_scaler.pkl')
joblib.dump(feature_cols_B, f'{MODELS_DIR}strategy_B_lag_feature_names.pkl')
print(f'Features: {X_train.shape[1]} | Train rows: {X_train.shape[0]:,} | mean ≈ {X_train.mean():.4f}')

## 4. XGBoost Grid Search

In [ ]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f'scale_pos_weight = {pos_weight:.1f}')

param_grid = [{'max_depth': d, 'learning_rate': lr}
              for d in [3, 4, 6] for lr in [0.05, 0.1]]

best_val_auprc, best_params, best_model = 0.0, None, None

for params in param_grid:
    m = xgb.XGBClassifier(
        n_estimators=500, max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        scale_pos_weight=pos_weight, eval_metric='aucpr',
        early_stopping_rounds=20, random_state=RANDOM_SEED, n_jobs=-1,
    )
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_auprc = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
    print(f"depth={params['max_depth']} lr={params['learning_rate']} val AUPRC={val_auprc:.4f}")
    if val_auprc > best_val_auprc:
        best_val_auprc, best_params, best_model = val_auprc, params, m

print(f'Best: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 5. Evaluate and Log

In [ ]:
val_probs  = best_model.predict_proba(X_val)[:, 1]
test_probs = best_model.predict_proba(X_test)[:, 1]

threshold    = select_threshold(y_val, val_probs)
val_metrics  = compute_all_metrics(y_val,  val_probs,  threshold)
test_metrics = compute_all_metrics(y_test, test_probs, threshold)

print(f'Test AUC-ROC: {test_metrics["auc_roc"]:.4f} | AUPRC: {test_metrics["auprc"]:.4f} | F1: {test_metrics["f1"]:.4f}')

log_results('B', 'XGBoost', 'Strategy_B', val_metrics, test_metrics, best_params)

joblib.dump(best_model, f'{MODELS_DIR}xgb_condition_B.pkl')
print(f'Saved | Features: {best_model.n_features_in_}')